# Семинар 1: знакомство с NumPy
### Фундамент, на котором строится весь курс по машинному обучению

## Зачем это всё нужно?

Прежде чем открывать документацию и разбирать синтаксис, договоримся, зачем мы вообще тратим на это первое занятие.

**Машинное обучение — это по существу линейная алгебра плюс немного статистики.** Любая модель — линейная регрессия, kNN, дерево решений, нейросеть — в конечном счёте сводится к операциям над числовыми массивами: векторами, матрицами, тензорами.

- Датасет из $n$ объектов с $d$ признаками — это матрица $X \in \mathbb{R}^{n \times d}$.
- Предсказание линейной модели — скалярное произведение: $\hat{y} = w^\top x + b$.
- Шаг градиентного спуска — покомпонентное обновление вектора весов: $w \leftarrow w - \eta \nabla L(w)$.
- Чёрно-белая картинка $28\times28$ для нейросети — просто матрица (или вектор из 784 чисел после `flatten`).

`numpy` — библиотека, которая умеет делать такие операции быстро, компактно и без циклов на чистом Python. Всё, что вы разберёте сегодня, будет использоваться на каждом следующем занятии: в регрессии, в градиентном спуске, в kNN, в PCA, в нейросетях.

### Маленькая демонстрация «на затравку»

Посчитаем сумму квадратов миллиона чисел двумя способами — обычным Python-циклом и через numpy — и сравним время работы:

In [ ]:
import numpy as np
import time

n = 1_000_000
data = list(range(n))
arr = np.arange(n)

start = time.time()
result_loop = sum(x ** 2 for x in data)
time_loop = time.time() - start

start = time.time()
result_numpy = np.sum(arr ** 2)
time_numpy = time.time() - start

print(f"Python-цикл: {time_loop:.4f} сек")
print(f"NumPy:       {time_numpy:.4f} сек")
print(f"Ускорение:   {time_loop / time_numpy:.0f}x")

Разница уже на этом простом примере впечатляет — а на матричном умножении, которое мы увидим в конце занятия, она будет ещё разительнее.

Причина в том, что numpy хранит данные в одном непрерывном блоке памяти фиксированного типа и выполняет операции векторизованно — на уровне C, а не интерпретируемого Python-байткода, да ещё и использует под капотом оптимизированные библиотеки линейной алгебры (BLAS/LAPACK).

### Мини-пример: во что превратится весь курс

Вот как выглядит предсказание логистической регрессии — одного из базовых алгоритмов классификации, который мы скоро разберём подробно — на numpy.

In [ ]:
# Логистическая регрессия: y_hat = sigmoid(w^T x + b)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w = np.array([0.5, -0.2, 0.1])   # веса модели
x = np.array([2.0, 1.0, 3.0])    # признаки одного объекта
b = 0.1                          # смещение (bias)

z = w @ x + b        # скалярное произведение вместо ручного цикла
y_hat = sigmoid(z)
print("z =", z, " y_hat =", y_hat)

Та же связка «взвешенная сумма признаков + нелинейная функция» — это ядро не только логистической регрессии, но и SVM, а в более сложном виде — и нейросетей (этим займётесь на других курсах, у нас в фокусе классические алгоритмы и бустинги). Базовая операция — умножение матрицы на вектор — везде одна и та же, и именно поэтому мы начинаем курс с numpy: это общий язык почти всех тем впереди.

Дальше — по порядку: сначала коротко про Jupyter, а затем сам numpy.

## Jupyter notebook

Полная документация: https://devpractice.ru/python-lesson-6-work-in-jupyter-notebook/




## numpy

- документация: http://www.numpy.org/

Библиотека numpy является удобным инструментом для работы с многомерными массивами с возможностью векторизации вычислений. Рассмотрим базовые вещи, которые можно делать с помощью нее.

In [ ]:
import numpy as np

Создание массивов

In [ ]:
vec = np.array([[1, 2.6], [3, 4], [5, 6]])

In [ ]:
vec

In [ ]:
print(vec)

С чем мы работаем?

In [ ]:
vec.dtype

In [ ]:
type(vec)

Размер массива:

In [ ]:
vec.shape

Число осей:

In [ ]:
vec.ndim

In [ ]:
vec.size

💡 **Где это используется в ML:** shape массива — то, что вы проверяете первым делом при работе с любыми данными. Табличный датасет из $n$ объектов и $d$ признаков — это матрица shape `(n, d)`, с ней работают и kNN, и деревья решений, и бустинг. Матрица попарных расстояний между всеми объектами (понадобится нам в kNN) — тоже двумерный массив, но уже shape `(n, n)`. 

In [ ]:
# Специальные конструкторы
z = np.zeros((2, 3))
o = np.ones((3, 2))
full = np.full((2, 2), fill_value=7)
eye = np.eye(4)              # единичная матрица
diag = np.diag([10, 20, 30]) # диагональная матрица
# Числовые последовательности
ar = np.arange(0, 10, 2)     # [0, 2, 4, 6, 8]
lin = np.linspace(0, 1, 5)   # 5 точек от 0 до 1 включительно

In [ ]:
lin

In [ ]:
diag

Агрегации и статистики

У некоторых функций бывает параметр `axis`, который позволяет применить эту функцию по разным осям - в данном случае, по строкам или столбцам:

In [ ]:
vec

In [ ]:
np.sum(vec)

In [ ]:
np.sum(vec, axis=0)

In [ ]:
np.sum(vec, axis=1)

In [ ]:
vec.sum()

In [ ]:
rng = np.random.default_rng(seed=42)
rng = rng.integers(0, 10, size=(3, 4))

In [ ]:
# Рекомендуется использовать новый генератор случайных чисел
rng = np.random.default_rng(seed=42)
A = rng.integers(0, 10, size=(3, 4))
print(A)
print("mean:", A.mean(), "std:", A.std(), "min:", A.min(), "max:", A.max())

# Индексы экстремумов
print("argmax flat:", A.argmax(), "-> индекс в форме:", np.unravel_index(A.argmax(), A.shape))
print("argmax по столбцам:", A.argmax(axis=0))

💡 **Где это используется в ML:** среднее и стандартное отклонение по столбцам — первый шаг предобработки почти любого датасета: **стандартизация (z-score)** $x' = \dfrac{x - \mu}{\sigma}$. Без неё градиентный спуск может сходиться медленно, если признаки имеют разный масштаб (скажем, «возраст» от 0 до 100 и «доход» от 0 до 1 000 000).

Изменение формы, транспонирование, конкатенация

Размеры массивов можно менять:

In [ ]:
vec

In [ ]:
vec.reshape(2, 3)

In [ ]:
vec.reshape(-1, 3)

In [ ]:
vec.reshape(2, -1)

In [ ]:
vec.reshape(6, 1)

In [ ]:
vec.reshape(5, 1)

In [ ]:
B = np.arange(12)
print("B:", B)
print("reshape 3x4:\n", B.reshape(3, 4))
print("ravel (view если возможно):", B.reshape(3,4).ravel())
print("flatten (всегда копия):", B.reshape(3,4).flatten())

In [ ]:
C = np.arange(6).reshape(2, 3)
print("C:\n", C)
print("C.T:\n", C.T)
print("swapaxes 0<->1:\n", np.swapaxes(C, 0, 1))

In [ ]:
# Склейки и разбиения
X1 = np.arange(6).reshape(2, 3)
X2 = np.arange(100, 106).reshape(2, 3)
print("concatenate по оси 1:\n", np.concatenate([X1, X2], axis=1))
print("vstack:\n", np.vstack([X1, X2]))
print("hstack:\n", np.hstack([X1, X2]))

In [ ]:
X1

In [ ]:
X2

In [ ]:
Y = np.arange(10)
print("split на 3 части:", np.array_split(Y, 3))  # части могут быть неравными

In [ ]:
# repeat и tile
v = np.array([1, 2, 3])
print("repeat элементов:", np.repeat(v, [1, 2, 3]))  # [1, 2,2, 3,3,3]
print("tile в 2 строки:\n", np.tile(v, (2, 1)))

💡 **Где это используется в ML:** `reshape`/`flatten` нужны каждый раз, когда исходные данные пришли не в виде плоской таблицы «объекты × признаки» — например, признаки объекта собраны многомерным блоком, и его нужно развернуть в один вектор, прежде чем отдать классическому алгоритму. `.T` буквально входит в формулу линейной регрессии $\hat\beta = (X^\top X)^{-1} X^\top y$, которую мы разберём ниже.

Индексирование:

In [ ]:
vec

In [ ]:
vec[:, 1]

In [ ]:
vec[2, :]

In [ ]:
vec[1:2, 0]

In [ ]:
y = np.arange(10)
print("y:", y)
print("y[0], y[-1]:", y[0], y[-1])
print("y[2:7:2]:", y[2:7:2])   # срез с шагом
print("y[::-1]:", y[::-1])     # разворот

In [ ]:
# Булевы маски
M = np.arange(1, 13).reshape(3, 4)
mask = M % 2 == 0
print("mask:\n", mask)
print("Только четные:\n", M[mask])

In [ ]:
# Изменение по маске
M2 = M.copy()
M2[M2 < 5] = 0
print("M2:\n", M2)

In [ ]:
# Фэнси индексация (индексы-списки/массивы)
rows = [0, 2]
cols = [1, 3]
print("Выбор элементов М[rows, cols]:", M[rows, cols])  # (0,1) и (2,3)

In [ ]:
# Виды vs копии:
a = np.arange(5)
print(a)
view = a[::2] # срез — представление (view)
view[0] = 999
print("a изменился через view:", a)

copy_idx = a[[0, 2, 4]] # фэнси — копия
copy_idx[0] = -123
print("a не изменился через копию:", a)

💡 **Где это используется в ML:** булевы маски — стандартный способ фильтровать датасет: `X[y == 1]` выберет все объекты одного класса, `X[~np.isnan(X).any(axis=1)]` — выкинет строки с пропусками. Так на практике убирают выбросы и разбивают данные по условию — без единого цикла.

Арифметика, ufunc и broadcasting

In [ ]:
p = np.array([1, 2, 3])
q = np.array([10, 20, 30])

print("Сложение/вычитание:", p + q, p - q)
print("Умножение поэлементно:", p * q)
print("Деление поэлементно:", q / p)
print("Степень:", p ** 2)

In [ ]:
# Ufunc (универсальные функции) — поэлементные
r = np.linspace(-2, 2, 5)
print("r:", r)
print("exp(r):", np.exp(r))
print("log(|r|+1):", np.log(np.abs(r) + 1))
print("sin(r):", np.sin(r))

In [ ]:
# np.where — условное поэлементное ветвление
print(r)
relu = np.where(r > 0, r, 0)
print("ReLU(r):", relu)

In [ ]:
# Broadcasting: вычитание среднего из столбцов
rng = np.random.default_rng(seed=42)
X = rng.normal(size=(4, 3))
col_means = X.mean(axis=0)         # shape (3,)
X_centered = X - col_means         # (4, 3) - (3,) = (4, 3)
print("X:\n", X)
print("col_means:", col_means)
print("X_centered:\n", X_centered)

Обычно центрирование дополняют делением на стандартное отклонение — получается полноценная z-score стандартизация (та самая формула из примечания выше):

In [ ]:
col_stds = X.std(axis=0)
X_standardized = (X - col_means) / col_stds
print("Среднее после стандартизации (≈0):", X_standardized.mean(axis=0))
print("Std после стандартизации (≈1):     ", X_standardized.std(axis=0))

НО!

In [ ]:
A = np.ones((2, 3))    # shape (2, 3)
B = np.ones((4,))    # shape (4, ) - НЕСОВМЕСТИМО!

try:
    result = A + B     # ОШИБКА!
except ValueError as e:
    print(f"Ошибка: {e}")

In [ ]:
# Пример с добавлением осей
a = np.arange(3).reshape(3, 1)     # shape (3,1)
b = np.arange(4).reshape(1, 4)     # shape (1,4)
print("a:\n", a)
print("b:\n", b)
print("a + b (broadcast до 3x4):\n", a + b)

Работа с NaN и Inf


In [ ]:
W = np.array([1.0, np.nan, 2.0, np.inf, -np.inf])
print("W:", W)
print("isnan:", np.isnan(W))
print("isfinite:", np.isfinite(W).any())

In [ ]:
# Замена NaN на среднее
Z = rng.normal(size=(5, 3))
mask = rng.random((5, 3)) < 0.3
Z[mask] = np.nan
col_means = np.nanmean(Z, axis=0)
Z_filled = np.where(np.isnan(Z), col_means, Z)
print("Z с NaN:\n", Z)
print("Z без NaN:\n", Z_filled)

In [ ]:
col_means

In [ ]:
np.nanmean(Z, axis=0)

💡 **Где это используется в ML:** реальные данные почти никогда не бывают чистыми — пропуски, выбросы, бесконечности встречаются постоянно. `np.isnan`, `np.nanmean`, `np.where` — базовый набор инструментов для быстрой очистки данных на этапе разведочного анализа (EDA), ещё до того как данные попадут в модель.

Случайные числа: выборки и перестановки

In [ ]:
rng = np.random.default_rng(123)     # фиксируем сид для воспроизводимости

# Перестановка и выборка
arr = np.arange(10)
print("перемешанный arr:", rng.permutation(arr))
print("выбор 5 элементов без возвращения:", rng.choice(arr, size=5, replace=False))
print("выбор с весами:", rng.choice(arr, size=5, p=np.repeat(0.1, 10)))

# Простая симуляция: 100 бросков игрального кубика
dice = rng.integers(1, 7, size=100)
print("среднее значение кубика:", dice.mean())

💡 **Где это используется в ML:** `rng.permutation` и `rng.choice` — основа разбиения на train/test, кросс-валидации и бутстрэпа (например, в случайном лесе). Фиксированный `seed` делает эксперимент воспроизводимым — это важно и для домашних заданий, и для настоящих исследований.


Линейная алгебра

In [ ]:
# Вектор-матрица-умножение
A = rng.normal(size=(3, 3))
x = rng.normal(size=3)
y = A @ x  # эквивалент np.dot(A, x)
print("A:\n", A)
print("x:", x)
print("A @ x:", y)

# Решение линейной системы A x = b
b = rng.normal(size=3)
sol = np.linalg.solve(A, b)
residual = np.linalg.norm(A @ sol - b)
print("Решение:", sol, "Остаток:", residual)

# Собственные значения (для квадратной матрицы)
w, V = np.linalg.eig(A)
print("Собственные значения:", w)

# Нормы
print("||x||_2:", np.linalg.norm(x), "||A||_F:", np.linalg.norm(A, 'fro'))

Сортировка, argsort и top-k

In [ ]:
S = rng.integers(0, 100, size=12)
print("S:", S)
print("отсортировано:", np.sort(S))
print("индексы сортировки:", np.argsort(S))

# Быстрый top-k (порядок внутри k не гарантирован)
k = 3
idx_topk = np.argpartition(S, -k)[-k:]
topk_values = S[idx_topk]
# Если нужно отсортировать убыванием:
topk_sorted_idx = idx_topk[np.argsort(S[idx_topk])[::-1]]
print("top-3 значения:", S[topk_sorted_idx], "индексы:", topk_sorted_idx)

### Пример: ближайшие соседи (kNN) без единого цикла

Классический трюк: посчитать расстояния от одной точки до всех объектов датасета с помощью broadcasting, а затем через `argsort` найти $k$ ближайших:

In [ ]:
rng = np.random.default_rng(1)
X_data = rng.normal(size=(20, 2))       # 20 точек на плоскости
query_point = np.array([0.0, 0.0])

distances = np.sqrt(np.sum((X_data - query_point) ** 2, axis=1))  # broadcasting!
k = 3
nearest_idx = np.argsort(distances)[:k]
print("Индексы ближайших соседей:", nearest_idx)
print("Расстояния до них:", distances[nearest_idx])

Это и есть ядро алгоритма kNN: `X_data - query_point` работает благодаря broadcasting, а `argsort` даёт ранжирование по расстоянию. Через пару занятий вы соберёте из этого целый классификатор.

Почему вообще используют `numpy`?

Производительность: векторизация vs Python-цикл

In [ ]:
n = 300
A = np.random.rand(n, n)
B = np.random.rand(n, n)

In [ ]:
%%time
C = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        for k in range(n):
            C[i, j] += A[i, k] * B[k, j]

In [ ]:
%%time
C = A @ B

Итог: там, где это возможно, предпочитайте векторизованные операции numpy явным циклам на Python — и по скорости, и по читаемости кода. Это будет негласным правилом на протяжении всего курса.

### Задания для самостоятельного решения



1.Создайте массив от 0 до 15 включительно. Преобразуйте его в матрицу 4x4. Поменяйте местами 2-ю и 4-ю строки, затем 1-й и 3-й столбцы (без циклов).

2.Создайте массив 8x8 со значениями 0 и 1 в шахматном порядке. Верхний левый элемент должен быть 1.

3.Дан a = np.arange(1, 101). Получите массив квадратов только для элементов, кратных 3 или 5, а остальные замените на -1 (используйте np.where или маски).

4.Сгенерируйте два массива X1 и X2 размерностями (5, 3) из N(0,1). Склейте их по вертикали, затем разделите обратно на две части без копирования данных, если это возможно. Подсказка: обратите внимание на то, какие операции создают представления (view), а какие — копии.

5.Сгенерируйте матрицу X (100 x 3) из N(0, 5). Приведите каждый столбец к нулевому среднему и единичному стандартному отклонению. Проверьте, что среднее каждой колонки ~ 0 и std ~ 1.


6.Дана матрица точек X размером (50, 2). Посчитайте матрицу попарных евклидовых расстояний размером (50, 50) без единого цикла (подсказка: используйте broadcasting с newaxis, как в примере про kNN выше).

In [ ]:
vec1 = np.arange(16)
mat1 = vec1.reshape(4, 4)
mat2 = mat1[[0, 3, 2, 1]]
mat3 = mat2[:, [2, 1, 0, 3]]

In [ ]:
vec1

In [ ]:
mat1

In [ ]:
mat2

In [ ]:
mat3

2.Создайте массив 8x8 со значениями 0 и 1 в шахматном порядке. Верхний левый элемент должен быть 1.

In [ ]:
z = np.zeros((8, 8))


In [ ]:
z

In [ ]:
vec1 = np.arange(64).reshape(8,8)
vec2 = np.where(vec1%2==1, 1, 0)


In [ ]:
vec1

In [ ]:
vec2